> 📓 **Lesson 1.8 — Part 2 of 4: Data Quality — Missing Data, Duplicates & Impossible Values**
>
> This notebook was split out of the original single `eda_basic.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.8.
>
> Other notebooks in this set: `Part_1_descriptive_statistics.ipynb`, `Part_3_data_transformation.ipynb`, `Part_4_reading_writing_data.ipynb`

# Lesson 1.8: EDA Basic

Welcome to Exploratory Data Analysis. This notebook takes one raw, messy business file and walks the
full path: understanding its structure, cleaning it, transforming it, and moving it in and out of
files.

**Structure — the four learning outcomes, in order:**
* **Part 1: Descriptive Statistics** — *summarise* a dataset: shape, data types, distributions.
* **Part 2: Data Quality** — *handle* the messy reality: missing values, duplicates, impossible values.
* **Part 3: Data Transformation** — *transform* for analysis: mapping, labels, strings, categories, dates.
* **Part 4: Reading & Writing Data** — *read and write* CSV, JSON, Excel, databases.

**For Learners:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 150 minutes.** One messy file, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Descriptive Statistics | **Summarise** a dataset: shape, dtypes, distributions | 33 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Quality | **Handle** missing values, duplicates, impossible values | 42 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Data Transformation | **Transform**: mapping, labels, strings, categories, dates, grouping | 35 min |
> | **Part 4** | Reading & Writing Data | **Read and write** CSV, JSON, Excel, databases | 15 min |
>
> **The spine:** we work on one file, `data/cafe_june_raw.csv`, from start to finish. Each section
> improves the same `clean` table, and Part 4 saves it. Small hand-built tables appear alongside it
> as *drills* — they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`;
> the Appendix at the end is self-study.


### The business problem

> **The Daily Grind** is a four-outlet café chain in Singapore. Revenue has been flat for two
> quarters, and the owner has to decide whether to renew the Marina Bay lease. She asks her
> assistant to send you the sales data. What arrives is a **raw till export**: one row per outlet,
> per day, per part of the day, straight out of the point-of-sale system, untouched.
>
> Nobody can answer the owner's question from this file yet. Today's job is to make it
> answerable — and to be able to say *why* every number in it can be trusted.

This is the first of three lessons on the same problem:

| Lesson | The question | What you do |
|---|---|---|
| **1.8 — today** | **Can I trust this data?** | clean one month of the raw export |
| 1.9 | What is the pattern? | 18 months, cleaned: time, joins, grouping |
| 1.10 | How do I make them act? | one chart, one slide, one decision |


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Load the two toolkits we need. `pd` and `np` are just short nicknames so we can
#    type `pd.something` instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np


In [ ]:
# 👉 Load the dataset we will use all session: June 2025's till export from four cafés.
#    `read_csv` reads a comma-separated text file into a DataFrame -- a table with named columns.
#    pandas already treats an empty field, "NA" and "n/a" as missing.
raw = pd.read_csv("../data/cafe_june_raw.csv")

raw


### 🎬 Why this matters — before you trust a single number

Run the next three cells. The chain has **four** cafés and its busiest shift takes about \$1,000.


In [ ]:
# 👉 `.value_counts()` counts how many rows have each value. How many cafés do you count?
raw["outlet"].value_counts()


In [ ]:
# 👉 The same question of the daypart column. There are three parts to a trading day.
raw["daypart"].value_counts()


In [ ]:
# 👉 Sort the takings column and look at the two ends. `.dropna()` skips the blank cells,
#    because a sort cannot compare text with a blank -- which is itself a clue.
#    `.iloc[[0, -1]]` takes the first and last rows of the sorted result.
raw["revenue_raw"].dropna().sort_values().iloc[[0, -1]]


**Three problems, in three lines of output.**

1. **Twelve spellings for four cafés.** `Raffles Place`, `raffles place`, `RAFFLES PLACE`,
   `Raffles Pl.`… Group by outlet today and you get twelve cafés, four of which are the same shop.
2. **Nine labels for three dayparts** — `Morning`, `morning`, `AM`, and so on.
3. **The revenue column is not a number.** Sorted, the "smallest" value is `" 1,006.71 "` and the
   "largest" is `"S$94.41"`, because pandas is comparing them as **text**: a space sorts before a
   digit, and the letter `S` sorts after every digit. Sorted as text, \$98,000 loses to \$99.

Any average, chart or model built on this file is wrong before you start. Worse, none of it would
*look* wrong: it would produce numbers, with decimal places, and nobody in the meeting would know.

Part 1 is the routine that finds problems like these in about two minutes.


---

## Part 2: Data Quality — Missing Data, Duplicates & Impossible Values

**Learning outcome 2:** *Handle missing values, duplicates, and outliers using appropriate Pandas
methods.*

**Goal:** work through the five problems on the Part 1 to-do list. Every fix follows the same four
beats — **find it → decide → apply → verify** — and that habit is worth more than any single method.

⏱️ ~42 min including Group Exercise 2


### 2.0: One type fix first

We cannot check whether the takings are plausible while they are **text**. This is the one line of
string-cleaning we need up front; the full toolkit is section 3.3.


In [ ]:
# 👉 `.copy()` makes an independent table, so `raw` stays as the untouched original --
#    which is what lets us compare "before and after" at the end of the session.
clean = raw.copy()

# 👉 Strip out everything that is not a digit, a dot or a minus sign: "S$1,240.50" -> "1240.50".
#    `regex=True` says "the thing I am searching for is a pattern, not literal text".
#    `to_numeric` then turns that text into a real number; `errors="coerce"` puts NaN
#    wherever the text could not be read as a number at all.
clean["revenue_sgd"] = pd.to_numeric(
    clean["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

clean[["revenue_raw", "revenue_sgd"]].head()


In [ ]:
# 👉 NOW `.describe()` can see the money column. Read the min and the max.
clean["revenue_sgd"].describe()


> **There it is.** A minimum of **-999** and a maximum of **98,000**, in a business whose busiest
> shift takes about \$1,000. Neither is a real number:
>
> - **-999** is a *sentinel* — the old till wrote it when a shift failed to close off. It means
>   "no reading", not "minus nine hundred and ninety-nine dollars".
> - **98,000** is a keying error: someone typed `98000` for a shift that took about \$980.
>
> Both were completely invisible while the column was text. This is why move 4 of the first look
> (`.dtypes`) exists.


### 2.1: Handling Missing Data

Missing data shows up as `NaN` (Not a Number) or `None`.


**Step 1 — find the holes on our own dataset.**


In [ ]:
# 👉 `.isna()` marks every cell True/False for "is this missing?". Chaining `.sum()`
#    counts the Trues per column, because True counts as 1.
clean.isna().sum()


In [ ]:
# 👉 But look at `notes` more closely. Some cells say "N.A." or "-" -- which a human reads as
#    empty and pandas reads as ordinary text. Fake blanks are worse than real ones: they are
#    invisible to `.isna()`.
clean["notes"].value_counts(dropna=False)


In [ ]:
# 👉 Turn the fake blanks into real ones, so `.isna()` tells the truth from here on.
clean["notes"] = clean["notes"].replace(["N.A.", "-"], np.nan)

print("missing notes before: 332")
print("missing notes after: ", clean["notes"].isna().sum())


**Step 2 — decide, column by column.** There is no single right answer, only defensible ones:

| Column | Holes | Decision | Why |
|---|---|---|---|
| `notes` | 362 | leave as `NaN` | "no note" is genuinely no information; do not invent one |
| `manager_email` | 2 | leave as `NaN` | you cannot guess an email address |
| `staff_on_shift` | 5 | fill with the **median** | staffing is fairly consistent; a typical value is a fair guess |
| `items` | 3 | fill with the **median** | same reasoning |
| `revenue_sgd` | 3 | **wait** | the sentinels must go first, or the median is poisoned |
| `tickets` | 2 | **wait** | there are impossible values in here too |

The last two rows are the lesson. **Order matters**, and section 2.3 is where it bites.


In [ ]:
# 👉 Apply the two easy decisions. A dictionary lets you use a different filler per column.
#    `.median()` is the middle value, which ignores lopsided extremes -- safer than the mean.
clean = clean.fillna({
    "staff_on_shift": clean["staff_on_shift"].median(),
    "items": clean["items"].median(),
})

clean[["staff_on_shift", "items"]].isna().sum()


Drills on smaller data follow, so you can see each method in isolation.


In [ ]:
# 👉 A Series of numbers where one entry is missing.
float_data = pd.Series([1.2, -3.5, np.nan, 0])

float_data


In [ ]:
# 👉 `.isna()` answers 'is this one missing?' for every entry: True means missing.
float_data.isna()


The built-in Python `None` value is also treated as missing in pandas object columns.


In [ ]:
# 👉 Python's own `None` also counts as missing, alongside `np.nan`.
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data


In [ ]:
# 👉 Proof: both `np.nan` and `None` come back as True.
string_data.isna()


### Strategy 1: Dropping Missing Data (`dropna`)

The simplest strategy is to remove the rows or columns that contain missing values.


In [ ]:
# 👉 `.dropna()` returns a copy with the missing entries removed. The original is unchanged --
#    nothing in pandas edits in place unless you assign the result back.
float_data.dropna()


In [ ]:
# 👉 The manual version of the same thing: `notna()` marks the good rows, and putting that
#    True/False mask in square brackets keeps only the True ones.
float_data[float_data.notna()]


With DataFrames, `dropna` by default drops **any row** containing **any** missing value.


In [ ]:
# 👉 A 4-row table with missing values scattered around.
demo_df = pd.DataFrame([[1., 6.5, 3.], [1., np.nan, np.nan],
                        [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])

demo_df


In [ ]:
# 👉 Default behaviour: drop a row if it has *any* missing value. Harsh -- only row 0 survives.
demo_df.dropna()


`how="all"` only drops rows where **every** value is missing.


In [ ]:
# 👉 Gentler: only drop a row where every single value is missing.
demo_df.dropna(how="all")


To drop **columns** instead of rows, pass `axis="columns"`.


In [ ]:
# 👉 Add a brand-new column that is entirely missing, so we have something to drop.
demo_df[4] = np.nan

demo_df


In [ ]:
# 👉 `axis="columns"` switches the target from rows to columns, so the all-missing column goes.
demo_df.dropna(axis="columns", how="all")


You can also set a **threshold**: keep only rows with at least `n` real values.


In [ ]:
# 👉 A 7-row, 3-column table of random numbers, then poke holes in it.
#    `.iloc[rows, columns]` selects by position, so :4 means "the first four rows".
rng = np.random.default_rng(seed=12345)
holes = pd.DataFrame(rng.standard_normal((7, 3)))
holes.iloc[:4, 1] = np.nan
holes.iloc[:2, 2] = np.nan

holes


In [ ]:
# 👉 With the default settings, most rows are gone.
holes.dropna()


In [ ]:
# 👉 `thresh=2` means 'keep the row if it has at least 2 real (non-missing) values'.
holes.dropna(thresh=2)


### Strategy 2: Filling Missing Data (`fillna`)

Instead of losing data, fill the holes with a constant or a calculated value.


In [ ]:
# 👉 `.fillna()` plugs every hole with a value instead of deleting the row.
holes.fillna(0)


You can specify a different fill value for each column:


In [ ]:
# 👉 Pass a dictionary to use a different filler per column: {column_name: fill_value}.
holes.fillna({1: 0.5, 2: 0})


**Forward / backward fill:** carry a neighbouring value into the gap. Common with time series.


In [ ]:
# 👉 `bfill` = backward fill: copy the next real value upwards into the gap.
holes.bfill()


In [ ]:
# 👉 `limit=2` stops the copying after 2 rows, so long gaps are not silently invented.
holes.bfill(limit=2)


**Imputation:** filling with the mean or median is a very common technique.


In [ ]:
# 👉 A small Series with two holes in it.
s_holes = pd.Series([1., np.nan, 3.5, np.nan, 7])

s_holes


In [ ]:
# 👉 'Imputation': fill the holes with the average of the values we do have.
#    `s_holes.mean()` is computed from the 3 real values only.
s_holes.fillna(s_holes.mean())


### 2.2: Handling Duplicates

Duplicate rows inflate every total built from them. The first look said 366 rows for a month that can
only contain 360 outlet-day-daypart combinations.


In [ ]:
# 👉 `.duplicated()` flags a row True if an identical row appeared earlier.
#    `.sum()` counts them.
clean.duplicated().sum()


In [ ]:
# 👉 Show the offending rows so you can eyeball them before deleting anything.
#    `keep=False` marks *both* copies, not just the later one, so they sit together.
clean[clean.duplicated(keep=False)].sort_values(["date_text", "outlet", "daypart"]).head(8)


In [ ]:
# 👉 Genuine full-row copies, so drop them. Assign back for the change to stick.
clean = clean.drop_duplicates()

clean.shape


360 rows. **Note the safer habit:** here the whole row was identical, so `drop_duplicates()` is
enough. In real exports the same shift is often sent twice with a *corrected* figure, and then the
rows are not identical — you have to say which columns identify a shift, and which copy to keep.

Check it explicitly:


In [ ]:
# 👉 A shift is identified by outlet + date + daypart. After de-duplicating there should be
#    exactly one row per combination -- but the outlet column is still spelled 12 ways,
#    so this check cannot pass until Part 3 fixes that. Try it and see.
clean.duplicated(subset=["date_text", "outlet", "daypart"]).sum()


> **Read that number carefully.** It is 0 — and that is *not* reassuring. Every "Marina Bay" row
> and every "marina bay" row look like different outlets to pandas, so a genuine double-submission
> under two spellings would slip straight through this check. **You cannot de-duplicate reliably
> until the key columns are standardised**, which is Part 3. Note it down and come back.


In [ ]:
# 👉 Build a table from a dictionary: each key becomes a column name, each list becomes a column.
dupes = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"], "k2": [1, 1, 2, 3, 3, 4, 4]})

dupes


In [ ]:
# 👉 For each row: 'have I already seen this exact row above?' True means it is a repeat.
dupes.duplicated()


In [ ]:
# 👉 Keep only the first appearance of each row and throw the repeats away.
dupes.drop_duplicates()


**Subset:** sometimes you only care about duplicates in specific columns.


In [ ]:
# 👉 Add a column of running numbers so we can see which rows survive the next steps.
dupes["v1"] = range(7)

dupes


In [ ]:
# 👉 `subset=` narrows the comparison: rows count as duplicates if column k1 matches,
#    even where v1 differs. This is the version you want for "same shift, sent twice".
dupes.drop_duplicates(subset=["k1"])


**Keep:** by default pandas keeps the first occurrence. Keep the last instead with `keep="last"`.


In [ ]:
# 👉 With a corrected re-submission, the LAST copy is usually the one you want.
dupes.drop_duplicates(subset=["k1", "k2"], keep="last")


### 2.3: Handling Impossible Values and Outliers

An **outlier** is a value far from the rest. Some are real and important; some are errors. The
question is never "is it extreme?" but **"is it possible?"**


**On our dataset:** four impossible-value problems, and they need different treatments.


In [ ]:
# 👉 Boolean filtering: build a True/False test, put it in the square brackets, and only
#    the True rows come back. `|` means OR.
impossible = clean[
    (clean["revenue_sgd"] < 0) | (clean["revenue_sgd"] > 20000) | (clean["tickets"] <= 0)
]

impossible[["outlet", "date_text", "daypart", "revenue_sgd", "tickets"]]


Eight rows, four different problems, four different right answers:

| What you see | What it means | Decision |
|---|---|---|
| `revenue_sgd` = **-999** (×4) | sentinel: the till failed to close off | not a number at all → `NaN`, then fill |
| `revenue_sgd` = **98,000** | keyed `98000` for about `980.00` | we cannot recover it → `NaN`, then fill |
| `tickets` = **0** with revenue > 0 | money taken, no receipts counted | impossible combination → `NaN`, then fill |
| `tickets` = **-4** | a negative count of customers | impossible → `NaN`, then fill |

Notice that all four end in `NaN`. **That is deliberate:** turning a wrong number into an explicit
hole is honest, and it hands the decision to the fill step where you have to state your reasoning.
Silently overwriting it with a plausible-looking number is how bad data survives.


In [ ]:
# 👉 `.mask(condition)` replaces values where the condition is True with NaN.
#    Read it as "hide the values I cannot believe".
clean["revenue_sgd"] = clean["revenue_sgd"].mask(
    (clean["revenue_sgd"] < 0) | (clean["revenue_sgd"] > 20000)
)
clean["tickets"] = clean["tickets"].mask(clean["tickets"] <= 0)

clean[["revenue_sgd", "tickets"]].describe()


Now the ranges are plausible — and we have created new holes on purpose. Eight shifts need a value:
the four sentinels, the mis-keyed \$98,000, and the three that were blank from the start.

**Fill them in the right order — sentinels out first, statistic second.** The next cell shows exactly
how much that order is worth, and the answer is more interesting than "a lot".


In [ ]:
# 👉 What the ordering is actually worth. Compare the statistic computed BEFORE masking
#    (with -999 and 98,000 still in the column) against the same statistic computed after.
#    (`raw.drop_duplicates()` first, so we are comparing like with like -- the duplicated
#     batch is already out of `clean`.)
before = pd.to_numeric(
    raw.drop_duplicates()["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True),
    errors="coerce",
)

print(f"median before masking: ${before.median():>8,.2f}   after: ${clean['revenue_sgd'].median():>8,.2f}")
print(f"mean   before masking: ${before.mean():>8,.2f}   after: ${clean['revenue_sgd'].mean():>8,.2f}")


> **Read those two lines carefully — they are the real lesson.**
>
> The **median** barely moves: \$456 → \$463. That is not luck; it is the whole reason a median is
> the safer default. It looks at the middle of the sorted values, so four absurd lows and one absurd
> high hardly shift it.
>
> The **mean** moves from \$742 to \$486. Had you filled with the mean in the wrong order, all eight
> filled shifts would have been **53% too high**, and the month's total would have been overstated by
> thousands.
>
> So the rule is not "the median gets poisoned" — it is: **mask first, then compute, because you
> cannot see from the outside which case you are in.** Today the median gave you a safety net. The
> next dataset might have thirty sentinels instead of four, and then even the median moves. Order the
> steps correctly and you never have to know.


In [ ]:
# 👉 Median, not mean, for exactly the reason above. (In Lesson 1.9 you will learn to fill per
#    outlet and daypart, which is finer and better; a single median is honest enough for now.)
clean = clean.fillna({
    "revenue_sgd": clean["revenue_sgd"].median(),
    "tickets": clean["tickets"].median(),
})

print("holes left in revenue_sgd:", clean["revenue_sgd"].isna().sum())
print("holes left in tickets:    ", clean["tickets"].isna().sum())


**Compare with the original to see what cleaning bought us:**


In [ ]:
# 👉 The same question asked of the dirty file and the clean one. This is the number that
#    would have gone into the owner's report.
dirty_total = pd.to_numeric(
    raw["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
).sum()

print(f"June revenue, raw file:   ${dirty_total:,.2f}")
print(f"June revenue, cleaned:    ${clean['revenue_sgd'].sum():,.2f}")
print(f"difference:               ${dirty_total - clean['revenue_sgd'].sum():,.2f}")


> **The raw file overstates June by more than \$90,000 — about 52%.** One mis-keyed shift did most
> of it, six duplicated rows added more, and four sentinels quietly pulled in the other direction.
>
> Every one of those was a two-line fix. None of them announced itself. A report built on the raw
> file would have told the owner her chain had its best month ever.

Drills on random data follow, where the mechanics are easier to see.


In [ ]:
# 👉 1000 rows of random numbers shaped like a bell curve, so most values sit near 0.
#    Values beyond about ±3 are rare by construction -- which makes them useful practice outliers.
spread = pd.DataFrame(np.random.default_rng(seed=12345).standard_normal((1000, 4)))

spread.describe()


**Detection:** find values more than 3 away from zero.


In [ ]:
# 👉 Grab one column, then keep only the entries more than 3 away from zero.
#    `.abs()` ignores the sign, so it catches both tails at once.
col = spread[2]

col[col.abs() > 3]


To find **any row** with an outlier in **any column**, use `.any(axis="columns")`.


In [ ]:
# 👉 Same test applied to the whole table. `.any(axis="columns")` asks, per row,
#    'was there at least one True?'
spread[(spread.abs() > 3).any(axis="columns")]


**Capping:** instead of removing outliers, pull them back to a threshold.


In [ ]:
# 👉 'Capping': pull extreme values back to the ±3 boundary instead of deleting the row.
#    `np.sign` keeps the direction (-1 or +1) so a low outlier becomes -3, not +3.
spread[spread.abs() > 3] = np.sign(spread) * 3

spread.describe()


**Removal:** or drop the rows entirely.


In [ ]:
# 👉 The other option -- 'trimming': keep only rows where every column is within bounds.
#    `~` flips True/False, so this reads "not (any column out of bounds)".
spread[~(spread.abs() > 2.9).any(axis="columns")].shape


### 🛠️ Group Exercise 2 — Data Quality (8 min)

Using the `practice` table below, fill the holes in column `b` with that column's median, leaving columns `a` and `c` untouched. *Hint:* `.fillna({...})` takes a dictionary.

*Expected:* column `b` has no holes left; `a` and `c` keep their original hole in row 4.

In [ ]:
# 👉 Practice data for the exercise: 6 rows of random numbers with holes punched in three
#    places. `.iloc[row, column]` selects one cell by position.
practice = pd.DataFrame(
    np.random.default_rng(seed=7).standard_normal((6, 3)).round(2), columns=["a", "b", "c"]
)
practice.iloc[0, 1] = np.nan
practice.iloc[2, 1] = np.nan
practice.iloc[4, [0, 1, 2]] = np.nan

practice


---

## ✅ Sample Solution

Try the exercise yourself first — this is *a* solution, not *the* solution. If your code reaches the same answer a different way, it is right.

**Fill the holes in column `b` with that column's median, leaving `a` and `c` untouched.**

In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A dictionary tells `.fillna()` which column gets which filler, so only `b` is touched.
#    `.median()` skips the NaNs itself, so it is the median of the real values.
practice.fillna({"b": practice["b"].median()})


In [ ]:
# 👉 Verify: column `b` should have no holes left, while `a` and `c` keep the hole in row 4.
practice.fillna({"b": practice["b"].median()}).isna().sum()


---

# ☕ Break — 10 minutes

**Where we are:** the numbers are now plausible and the duplicates are gone.
**Next up:** Part 3 — the text columns, the dates, and the answer the owner actually asked for.


📂 **Open** `Part_3_data_transformation.ipynb` to continue.